# Data Master: inspeção Delta read-only

Este notebook observa os mesmos snapshots sintéticos processados por Airflow/Spark em MinIO. Ele não escreve no lakehouse e não participa do caminho crítico do pipeline.

In [ ]:
from jobs.presentation.jupyter_delta_path import (
    create_presentation_session,
    prepare_presentation_views,
    run_prepared_gold_query,
    validate_presentation_session,
)

spark = create_presentation_session()
validate_presentation_session(spark)

## Preparar views estáveis

Os paths físicos são resolvidos uma vez pelo helper. As consultas seguintes usam apenas nomes de views.

In [ ]:
inspection = prepare_presentation_views(spark)
inspection

In [ ]:
spark.sql("""
SELECT 'Bronze' AS camada, COUNT(*) AS linhas FROM bronze_transacoes
UNION ALL
SELECT 'Raw Vault', COUNT(*) FROM raw_hub_transacao
UNION ALL
SELECT 'Gold', COUNT(*) FROM gold_transacoes_por_dia
""").show(truncate=False)

## Consulta Gold preparada

A consulta abaixo usa uma view curta e apresenta apenas agregados sintéticos.

In [ ]:
run_prepared_gold_query(spark).show(truncate=False)